# Calculate rain-use-efficiency (RUE)

In [2]:
import rasterio
import numpy as np
import pandas as pd

In [3]:
# ---------- Setup ----------

# Define years and variables to calculate RUE for
years = list(range(2002, 2024))  # 2002-2023 inclusive

# Paths to input and output files
tif_paths = {y: f"/Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_{y}.tif" for y in years}
parquet_paths = {y: f"/Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_{y}.parquet" for y in years}
tif_output_path = "/Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/"
parquet_output_path = "/Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/"

In [4]:
# ---------- Raster (.tif) processing ----------
for y in years:
    with rasterio.open(tif_paths[y]) as src:
        # Get band names/descriptions to find Npp and pr_sum
        band_names = src.descriptions
        npp_idx = band_names.index("Npp") + 1      # rasterio bands are 1-indexed
        pr_idx = band_names.index("pr_sum") + 1

        npp = src.read(npp_idx).astype("float32")
        pr_sum = src.read(pr_idx).astype("float32")

        # Avoid division by zero
        rue = np.where(pr_sum != 0, npp / pr_sum, np.nan).astype("float32")

        # Prepare output profile with one extra band
        profile = src.profile.copy()
        profile.update(count=src.count + 1, dtype="float32")

        out_path = f"{tif_output_path}stack_wgs84_{y}_rue.tif"
        with rasterio.open(out_path, "w", **profile) as dst:
            # Copy existing bands
            for i in range(1, src.count + 1):
                dst.write(src.read(i), i)
                dst.set_band_description(i, band_names[i - 1])
            # Write new RUE band
            dst.write(rue, src.count + 1)
            dst.set_band_description(src.count + 1, "rue")

    print(f"[TIF] {y}: RUE band added -> {out_path}")

/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_6232/102021228.py:9: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  npp = src.read(npp_idx).astype("float32")
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_6232/102021228.py:10: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  pr_sum = src.read(pr_idx).astype("float32")
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_6232/102021228.py:13: RuntimeWarning: divide by zero encountered in divide
  rue = np.where(pr_sum != 0, npp / pr_sum, np.nan).astype("float32")
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_6232/102021228.py:23: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternati

[TIF] 2002: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2002_rue.tif
[TIF] 2003: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2003_rue.tif
[TIF] 2004: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2004_rue.tif
[TIF] 2005: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2005_rue.tif
[TIF] 2006: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2006_rue.tif
[TIF] 2007: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2007_rue.tif
[TIF] 2008: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2008_rue.tif
[TIF] 2009: RUE band added -> /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2009_rue.tif
[TIF] 2010: RUE band added -> /Users/Wanja/Documents/non-equilib

In [5]:
# ---------- Parquet (.parquet) processing ----------
for y in years:
    df = pd.read_parquet(parquet_paths[y])

    df["rue"] = np.where(df["pr_sum"] != 0, df["Npp"] / df["pr_sum"], np.nan)

    out_path = f"{parquet_output_path}table_wgs84_{y}_rue.parquet"
    df.to_parquet(out_path, index=False)

    print(f"[Parquet] {y}: RUE column added -> {out_path}")

[Parquet] 2002: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2002_rue.parquet
[Parquet] 2003: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2003_rue.parquet
[Parquet] 2004: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2004_rue.parquet
[Parquet] 2005: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2005_rue.parquet
[Parquet] 2006: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2006_rue.parquet
[Parquet] 2007: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2007_rue.parquet
[Parquet] 2008: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2008_rue.parquet
[Parquet] 2009: RUE column added -> /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2009_rue.